# Patron de Reflexion: un agente que se autocritica

Este notebook muestra el patron de **reflexion** (generar -> criticar -> revisar): un LLM produce una respuesta, otro rol del mismo modelo (el "revisor") la critica, y si encuentra un problema el modelo la corrige. Se repite hasta que el revisor la aprueba o se agotan los intentos.

Caso de uso: un agente que explica conceptos de fisica a un estudiante, y se autocorrige antes de entregar la respuesta final.

Las dos "voces" (explicador y revisor) son el mismo LLM con distintas instrucciones de sistema.

Requisitos: Ollama corriendo localmente con al menos el modelo `llama3.2:1b` descargado (ver `ayuda.txt`).

In [1]:
from reflection_agent import explicar, revisar, corregir, run_reflection
from display_helpers import show_trace

## 1. Sin reflexion (linea base)

Primero pedimos la explicacion una sola vez, sin ningun paso de revision. Con un modelo pequeno como `llama3.2:1b`, a veces la explicacion tiene errores o imprecisiones que pasan sin que nadie las revise.

In [2]:
concepto = "por que el periodo de un pendulo simple no depende de la masa"
model = "llama3.2:1b"

explicacion_base = explicar(concepto, model)
print(explicacion_base)

Excelente pregunta, estudiante. Un pendulo simple es un sistema que se mueve alrededor de un eje, y su movimiento se puede estudiar de manera muy sencilla. Cuando un pendulo simple se mueve, la masa que lo compone se siente por el centro de gravedad. Pero ¿por qué el periodo de un pendulo simple no depende de la masa?

La razón es que la fuerza de gravedad es la misma en todas las direcciones, y la masa es lo que nos da la cantidad de materia que forma el pendulo. Si aumentamos la masa del pendulo, la fuerza de gravedad que nos envuelve es mayor, lo que significa que la pendulo se mueve más lento. Pero el tiempo es la distancia que se pasa por la pendulo en un cierto período de tiempo, y si aumentamos la masa, la pendulo tardará más tiempo en completar esta distancia. Es como si aumentara la velocidad del tren, pero la distancia que recorre el tren es la misma. 

En otras palabras, la masa no afecta la velocidad con la que se mueve el pendulo, sino más bien la distancia recorrida por e

## 2. El ciclo paso a paso

El patron tiene tres funciones en `reflection_agent.py`:

- `explicar(concepto, model)` -- genera una primera explicacion.
- `revisar(concepto, explicacion, model)` -- el mismo modelo, con otra instruccion de sistema, actua como revisor: dice si la aprueba y por que no si la rechaza.
- `corregir(concepto, explicacion, comentarios, model)` -- vuelve a generar la explicacion, esta vez viendo los comentarios del revisor.

Vamos a correrlas a mano una vez, sobre la explicacion de la seccion anterior.

In [3]:
revision = revisar(concepto, explicacion_base, model)
print("Aprobada:", revision["aprobada"])
print(revision["comentarios"])

Aprobada: False
APROBADA

COMENTARIOS: Se requiere mayor claridad en la explicación. La masa es la cantidad de materia que forma el pendulo, y su aumento afecta la velocidad, no la distancia recorrida.


In [4]:
if not revision["aprobada"]:
    explicacion_corregida = corregir(concepto, explicacion_base, revision["comentarios"], model)
    print(explicacion_corregida)
else:
    print("El revisor la aprobo, no hace falta corregir.")

Entiendo mejor la crítica del revisor y gracias por la oportunidad de aclarar el concepto.

Aquí te presento una versión corregida:

Estimado revisor,

Me parece que la explicación anterior aún no es suficientemente clara para resolver la pregunta de por qué el periodo de un pendulo simple no depende de la masa.

La masa es la cantidad de materia que forma el pendulo. Cuando aumentamos la masa del pendulo, la fuerza de gravedad que nos envuelve es mayor, lo que significa que la pendulo se mueve más lento. Pero la distancia recorrida por el pendulo en un cierto período de tiempo no cambia, porque la pendulo se mueve a una velocidad constante. El tiempo es la distancia recorrida por el pendulo en un cierto período de tiempo, y si aumentamos la masa del pendulo, la pendulo tardará más tiempo en completar esta distancia.

En otras palabras, la masa no afecta la velocidad con la que se mueve el pendulo, sino más bien la distancia recorrida por el pendulo en un cierto período de tiempo. En o

## 3. El ciclo completo: `run_reflection`

`run_reflection` encadena estos tres pasos automaticamente hasta `max_rounds` veces, o hasta que el revisor aprueba. Devuelve la traza completa (cada ronda con su explicacion y su revision) para poder inspeccionar que paso.

In [5]:
trace = run_reflection(concepto, model=model, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

Excelente pregunta. En un pendulo simple, el momento de un cuerpo en movimiento está dado por la masa del cuerpo multiplicada por su velocidad lineal (m × v) y dividida por 2 (según la fórmula de la masa y la velocidad lineal). Esto nos da una constante, llamada momento angular (θ), que depende de la masa del cuerpo y su velocidad.

Sin embargo, el momento angular también depende del tiempo que el pendulo ha estado girando. Por lo tanto, el momento angular puede cambiar con el tiempo, lo que significa que el periodo del pendulo puede cambiar. 

Este fenómeno es conocido como "efecto de la resonancia" y puede ser explicado de diferentes maneras. Una forma sencilla es que cuando un cuerpo se mueve a un ritmo constante, su momento angular no cambia. Pero cuando se acelera o disminuye a un ritmo diferente, su momento angular puede aumentar, lo que altera su período. En el caso de un pendulo simple, esta alteración del momento angular con el tiempo puede llevar a un período que no sea constante. 

Por ejemplo, si se acelera el pendulo, su momento angular aumenta, lo que significa que su período se reduce. Por otro lado, si se disminuye su aceleración, su momento angular disminuye, lo que lleva a un período mayor. En este sentido, el período del pendulo no depende de la masa, sino más bien de la velocidad y la frecuencia con la que se mueve.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
La explicación proporcionada es precisa y clara, y se toman en cuenta las diferentes perspectivas de resolución del problema (efecto de la resonancia). Sin embargo, se puede mejorar aún más al enfatizar la importancia del momento angular en el pendulo simple y cómo su cambio con el tiempo afecta la constante momento angular. Por ejemplo, se puede mencionar que esta constante momento angular es independiente de la masa, pero depende del momento angular, lo que significa que cuando la masa disminuye, el momento angular aumenta y afecta a la constante momento angular.

### Ronda 1

...

### Ronda 2

Muchas gracias por la retroalimentación. Aquí te dejo la versión corregida:

La ecuación para el momento angular en un pendulo simple es M = m × v. Este momento angular es independiente de la masa y depende del momento angular, que se determina por la ecuación M = ω × θ.

Cuando se acelera o disminuye el pendulo, su velocidad angular cambia, lo que altera su momento angular. Por lo tanto, el momento angular no es constante; depende de la velocidad angular. Por ejemplo, si se acelera el pendulo a 2 rad/s, su momento angular aumentará a 2 × 2 = 4 rad/s^2.

En otras palabras, la constante momento angular (M) es independiente de la masa (m), ya que depende del momento angular (θ), que es determinado por la velocidad angular (v). Esto significa que, aunque la masa puede aumentar o disminuir, su momento angular sigue siendo el mismo.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

COMENTARIOS: El problema es que el momento angular no es independiente de la masa cuando se acelera o disminuye el pendulo, sino que depende del momento angular y de la velocidad angular.

### Ronda 3

Muchas gracias por la retroalimentación. Aquí te dejo la versión corregida:

La ecuación para el momento angular en un pendulo simple es M = m × v. Este momento angular es independiente de la masa y depende del momento angular, que se determina por la ecuación M = ω × θ.

Cuando se acelera o disminuye el pendulo, su velocidad angular cambia, lo que altera su momento angular. Por lo tanto, el momento angular no depende de la masa y no es variable. En otras palabras, el momento angular es constante y no cambia con la velocidad angular. Por ejemplo, si se acelera el pendulo a 2 rad/s, su momento angular aumentará a 2 × 2 = 4 rad/s^2.

En otras palabras, la constante momento angular (M) es independiente de la masa (m), ya que depende del momento angular (θ), que es determinado por la velocidad angular (v). Esto significa que, aunque la masa puede aumentar o disminuir, su momento angular sigue siendo el mismo.

Es importante destacar que, aunque la masa sigue siendo la misma, su momento angular cambia en consecuencia.

## 4. Varios conceptos

Probemos con un par de conceptos mas -- el codigo no cambia, solo el texto que le pasamos.

In [6]:
conceptos = [
    "por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo",
    "diferencia entre velocidad y aceleracion",
]

for c in conceptos:
    trace = run_reflection(c, model=model, max_rounds=3)
    show_trace(c, trace)

## Concepto: por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo

### Ronda 0

Excelente pregunta! En primer lugar, vamos a analizar el caso del objeto en caída libre. Cuando un objeto está en caída libre, se encuentra bajo la acción de la gravedad, que es una fuerza que opone el aumento del movimiento de un objeto con respecto al tiempo. La caída libre se refiere a la velocidad que una persona o un objeto cae sobre el suelo, que disminuye constantemente debido a la gravedad.

En este caso, cuando un objeto en caída libre cae desde la misma altura, su velocidad decreciente en función del tiempo, lo que lo hace más lentamente caer. Pero, ¿cómo se relaciona esto con el toque del suelo? La respuesta se encuentra en la ecuación de la caída libre: h = (1/2)gt², donde h es la altura, g es la gravedad y t es el tiempo.

Si el objeto en caída libre cae desde la misma altura que el objeto lanzado horizontalmente, su velocidad es la misma en ambos casos. Sin embargo, su altura de descenso varía con el tiempo. El objeto lanzado horizontalmente se encuentra bajo la acción de la gravedad solo hasta la altura de descenso de su lanzador, y luego se detiene porque ya no está bajo la acción de la gravedad. Por otro lado, el objeto en caída libre se encuentra bajo la acción de la gravedad toda su vida, lo que lo hace continuar moviéndose hacia abajo con una velocidad decreciente.

En consecuencia, el objeto lanzado horizontalmente alcanza el suelo antes de que el objeto en caída libre. Esto se debe a que el objeto en caída libre ha caído la mayor parte de la distancia desde el momento en que se lanzó, mientras que el objeto lanzado horizontalmente solo ha llegado a la altura de descenso del lanzador.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

COMENTARIOS: La explicación del profesor es clara y concisa, y se toman en cuenta las diferencias entre ambos casos (caída libre y lanzado horizontalmente). Sin embargo, se observa que el profesor se enfoca más en la ecuación de la caída libre y las propiedades físicas, sin mencionar específicamente la velocidad relativa o el concepto de "altura de descenso". Además, la explicación del profesor se limita al caso del lanzado horizontalmente, sin mencionar el caso del objeto en caída libre.

### Ronda 1

...

### Ronda 2

Gracias por el feedback del revisor. Aquí te presento la versión corregida:

Explica: diferencia entre velocidad y aceleración

Imagina que estás corriendo por la calle y quieres saber cuánto tiempo te toma recorrer una certaine distancia. La velocidad es el ritmo en que vas recorriendo la distancia, es decir, ¿cómo muchos pasos se pasan por el camino? La velocidad es la distancia recorrida por unidad de tiempo, en otras palabras, ¿cuántas pasos se pasan por el camino en un minuto? Por ejemplo, si vas a correr a 25 km/h, eso significa que vas a recorrer 5 kilómetros en un minuto. Si es que vas a correr en una carretera larga y no hay mucho obstáculo, la velocidad podría ser de 30 km/h, lo que significaría que vas a recorrer 3 kilómetros en un minuto.

Ahora, imagina que estás en un coche y quieres saber cuánto tiempo te toma recorrer la misma distancia. La aceleración es el ritmo en que vas aumentando la velocidad, es decir, ¿cuántos kilómetros por minuto aumentas la velocidad? La aceleración es la distancia recorrida por unidad de tiempo, en otras palabras, ¿cuántos kilómetros se recorren por minuto en un tiempo determinado? Por ejemplo, si te aceleras a 100 km/h en 5 minutos, eso significa que vas a recorrer 2 kilómetros por minuto. Si es que te aceleras en una carretera corta y no hay muchos obstáculos, la aceleración podría ser de 120 km/h, lo que significaría que vas a recorrer 3 kilómetros por minuto.

En resumen, la velocidad es la distancia recorrida por unidad de tiempo, mientras que la aceleración es la distancia recorrida por unidad de tiempo, pero con un ritmo de cambio. Es importante tener en cuenta que la velocidad y la aceleración están relacionadas, pero no son la misma cosa. La velocidad es un aspecto más importante, ya que implica un cambio en la velocidad, mientras que la aceleración es un aspecto más específico, que es el cambio en la velocidad.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

COMENTARIOS: La explicación es clara y concisa, pero la terminología puede ser un poco abrumadora para algunos estudiantes. Por ejemplo, "velocidad es el ritmo en que vas recorriendo la distancia" podría ser reemplazado por "velocidad es la distancia recorrida por unidad de tiempo". Además, se hace una distinción entre velocidad y aceleración, pero no es clara que la aceleración sea la distancia recorrida por unidad de tiempo, sino que es más una cuestión de ritmo de cambio.

### Ronda 3

Gracias por el feedback del revisor. Aquí te presento la versión corregida:

Explica: diferencia entre velocidad y aceleración

Imagina que estás corriendo por la calle y quieres saber cuánto tiempo te toma recorrer una certaine distancia. La velocidad es el ritmo en que vas recorriendo la distancia, es decir, cuántas pasos se pasan por el camino en un minuto. Por ejemplo, si vas a correr a 25 km/h, eso significa que vas a recorrer 5 kilómetros en un minuto. La aceleración es el ritmo en que vas aumentando la velocidad, es decir, cuántos kilómetros por minuto aumentas la velocidad.

La aceleración es la distancia recorrida por unidad de tiempo, en otras palabras, ¿cuántos kilómetros se recorren por minuto en un tiempo determinado? Por ejemplo, si te aceleras a 100 km/h en 5 minutos, eso significa que vas a recorrer 200 kilómetros en 10 minutos. En resumen, la velocidad es la distancia recorrida por unidad de tiempo, mientras que la aceleración es la distancia recorrida en un tiempo determinado, con un ritmo de cambio.

En otras palabras, la velocidad es la distancia que vas recorriendo en un tiempo determinado, y la aceleración es la cantidad de velocidad que vas aumentando en un tiempo determinado. En un contexto, la aceleración es importante para lograr un objetivo, como arrancar un coche o correr una carrera, pero no es la velocidad que se logra en sí misma.

## 5. Modelo pequeno vs modelo mas grande

`llama3.2:1b` es rapido pero poco confiable, y eso se nota **en los dos roles**: el explicador puede cometer errores, y el revisor puede no detectarlos (o rechazar explicaciones que si estaban bien). Un modelo mas grande, como `qwen3:1.7b`, suele ser mas consistente en ambos roles -- sin tocar el resto del codigo, solo cambia el parametro `model`.

In [7]:
model_grande = "qwen3:1.7b"
trace = run_reflection(concepto, model=model_grande, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

El periodo de un péndulo simple no depende de la masa porque la fórmula del periodo, $ T = 2\pi \sqrt{\frac{L}{g}} $, muestra que la masa se cancela en el cálculo. Aunque la masa participa en la fuerza restauradora ($ F = -mg \sin\theta $), su influencia se anula al resolver la ecuación de movimiento, ya que la aceleración es independiente de la masa. Además, el péndulo se comporta como un movimiento armónico lineal, donde la aceleración depende solo de la longitud del cable y la aceleración gravitacional, no de la masa. Por lo tanto, todos los péndulos de igual longitud, bajo la misma gravedad, tienen el mismo periodo, independientemente de su masa.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA:  
Ninguno

### Ronda 1

La explicación es correcta en el sentido de que el periodo de un péndulo simple no depende de la masa, ya que la fórmula $ T = 2\pi \sqrt{\frac{L}{g}} $ muestra que la masa se cancela en el cálculo. Sin embargo, es importante destacar que la masa no participa en la derivación de esta fórmula, ya que el movimiento del péndulo se describe mediante un sistema de ecuaciones que se resuelve sin necesidad de considerar la masa. Aunque la fuerza gravitacional está relacionada con la masa ($ F = -mg \sin\theta $), su influencia en la ecuación de movimiento se anula al resolver la ecuación armónica que describe el péndulo. Por lo tanto, el periodo depende solo de la longitud del cable ($ L $) y la aceleración gravitacional ($ g $), no de la masa. Esta independencia se debe a que el péndulo se comporta como un movimiento armónico lineal, donde la aceleración es independiente de la masa.

**Revision del critico -- Aprobada**

APROBADA: si/no  
COMENTARIOS: La explicación es correcta en el sentido de que la masa no afecta el periodo del péndulo, pero se podría mejorar explicando claramente que la fórmula se deriva de la ecuación diferencial del movimiento armónico lineal, donde la masa se anula al resolver la ecuación. Además, se debería destacar que la relación F = -mg sinθ es válida pero no influye en la solución del periodo debido a la naturaleza de los movimientos armónicos.


La idea central -- **un LLM que critica y corrige su propia salida antes de entregarla** -- se traslada a otros flujos de un curso o laboratorio:

- **Retroalimentacion de tareas**: el agente redacta un comentario sobre la respuesta de un estudiante, y un segundo paso revisa que la retroalimentacion sea justa y este bien fundamentada antes de mostrarsela.
- **Otros conceptos**: solo cambia el texto que le pasas a `run_reflection`, no hace falta tocar el codigo.
- **Otro criterio de revision**: cambia `REVISOR_PROMPT` en `reflection_agent.py` para que el revisor chequee otra cosa (ej. que la explicacion use el vocabulario correcto para el nivel del curso, o que incluya una analogia).
- **Combinarlo con herramientas**: se puede mezclar con el patron de `Tools` -- por ejemplo, un agente que ajusta una curva y *despues* reflexiona sobre si el modelo elegido tiene sentido fisico.